# 01 — Exploratory Data Analysis (EDA)

**Dataset:** DE-LU hourly electricity snapshot (processed)
`data/processed/de_lu_hourly.parquet`

**Coverage:** 2019-01-01 → 2026-09-06 (Europe/Berlin delivery days), hourly UTC.

This notebook explores wind generation, the official TSO forecast, forecast
errors, prices, missing data, and temporal structure. It only reads the
processed dataset; it never trains a model.

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / "data" / "processed").exists():
    ROOT = Path.cwd().parent

FIG = ROOT / "reports" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

DATA = ROOT / "data" / "processed" / "de_lu_hourly.parquet"
df = pd.read_parquet(DATA)

print("shape:", df.shape)
print("index:", df.index.min(), "->", df.index.max())
print("columns:", list(df.columns))

Matplotlib is building the font cache; this may take a moment.


shape: (67343, 37)
index: 2018-12-31 23:00:00+00:00 -> 2026-09-06 21:00:00+00:00
columns: ['delivery_date', 'local_hour', 'utc_offset_minutes', 'price_eur_mwh', 'load_forecast_mw', 'wind_onshore_forecast_mw', 'solar_forecast_mw', 'wind_offshore_forecast_mw', 'wind_onshore_actual_mw', 'solar_actual_mw', 'wind_offshore_actual_mw', 'price_source_interval_count', 'load_forecast_mw_source_quarter_count', 'wind_onshore_forecast_mw_source_quarter_count', 'solar_forecast_mw_source_quarter_count', 'wind_offshore_forecast_mw_source_quarter_count', 'wind_onshore_actual_mw_source_quarter_count', 'solar_actual_mw_source_quarter_count', 'wind_offshore_actual_mw_source_quarter_count', 'vre_forecast_mw', 'vre_actual_mw', 'wind_total_actual_mw', 'wind_total_forecast_mw', 'wind_forecast_error_mw', 'is_target_missing', 'price_is_quarter_mean', 'decision_time_utc', 'wind_total_actual_mw_lag24', 'wind_total_actual_mw_lag48', 'wind_total_actual_mw_lag168', 'wind_actual_decision_lag1', 'wind_actual_decision_

In [2]:
key_cols = [
    "price_eur_mwh", "load_forecast_mw",
    "wind_total_actual_mw", "wind_total_forecast_mw",
    "wind_onshore_actual_mw", "wind_offshore_actual_mw",
]
print("missing values:")
print(df[key_cols].isna().sum().to_string())
print("\nduplicate timestamps:", int(df.index.duplicated().sum()))
print("\nsplit counts:")
print(df["split"].value_counts().to_string())
print("\nnegative prices:", int((df["price_eur_mwh"] < 0).sum()))

missing values:
price_eur_mwh                0
load_forecast_mw           194
wind_total_actual_mw         1
wind_total_forecast_mw       3
wind_onshore_actual_mw       1
wind_offshore_actual_mw      0

duplicate timestamps: 0

split counts:
split
train         35064
validation    17544
test          14735

negative prices: 2501


## 1. Wind generation over time

In [3]:
daily = df["wind_total_actual_mw"].resample("D").mean()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(daily.index, daily.values, lw=0.7)
ax.set_title("Daily mean total wind generation (DE-LU)")
ax.set_ylabel("MW")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIG / "01_wind_generation_daily.png", dpi=110)
plt.close(fig)

print(daily.describe())

count     2807.000000
mean     14856.515748
std       9902.890809
min        278.104167
25%       6894.968750
50%      12522.520833
75%      20947.692708
max      49382.937500
Name: wind_total_actual_mw, dtype: float64


## 2. Hourly, daily, monthly and seasonal patterns

In [4]:
df["hour"] = df.index.hour
hourly_mean = df.groupby("hour")["wind_total_actual_mw"].mean()

fig, ax = plt.subplots(figsize=(11, 4))
sns.boxplot(data=df, x="hour", y="wind_total_actual_mw", ax=ax, showfliers=False)
ax2 = ax.twinx()
ax2.plot(hourly_mean.index, hourly_mean.values, color="crimson", marker="o", label="mean")
ax2.set_ylabel("mean (MW)")
ax.set_title("Hourly pattern of total wind generation")
ax.set_xlabel("Hour (UTC)")
fig.tight_layout()
fig.savefig(FIG / "02_hourly_pattern.png", dpi=110)
plt.close(fig)

print("hourly mean by hour (UTC):")
print(hourly_mean.round(1).to_string())

hourly mean by hour (UTC):
hour
0     15517.3
1     15320.7
2     15174.0
3     15138.6
4     15161.3
5     14911.3
6     14300.0
7     13821.3
8     13558.6
9     13570.3
10    13678.2
11    13772.1
12    13888.0
13    14107.3
14    14401.6
15    14784.0
16    15064.6
17    15243.9
18    15495.3
19    15888.2
20    16057.7
21    15992.8
22    15905.3
23    15738.7


In [5]:
df["month"] = df.index.month
month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
df["month_name"] = df["month"].map(dict(enumerate(month_names, start=1)))

fig, ax = plt.subplots(figsize=(12, 4))
sns.boxplot(data=df, x="month_name", y="wind_total_actual_mw", ax=ax,
            order=month_names, showfliers=False)
ax.set_title("Monthly distribution of total wind generation")
fig.tight_layout()
fig.savefig(FIG / "03_monthly_pattern.png", dpi=110)
plt.close(fig)

season_map = {12:"Winter",1:"Winter",2:"Winter",3:"Spring",4:"Spring",5:"Spring",
              6:"Summer",7:"Summer",8:"Summer",9:"Autumn",10:"Autumn",11:"Autumn"}
df["season"] = df["month"].map(season_map)

fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=df, x="season", y="wind_total_actual_mw", ax=ax,
            order=["Winter","Spring","Summer","Autumn"], showfliers=False)
ax.set_title("Seasonal distribution of total wind generation")
fig.tight_layout()
fig.savefig(FIG / "04_seasonal_pattern.png", dpi=110)
plt.close(fig)

print(df.groupby("season")["wind_total_actual_mw"].agg(["mean","median","std"]).round(1).to_string())

           mean   median      std
season                           
Autumn  15583.7  13772.2  10451.8
Spring  13653.1  11348.0   9695.0
Summer   9756.0   7916.2   7130.5
Winter  20882.6  19722.5  12237.6


## 3. Onshore vs offshore generation

In [6]:
monthly = df[["wind_onshore_actual_mw", "wind_offshore_actual_mw"]].resample("MS").mean()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(monthly.index, monthly["wind_onshore_actual_mw"], label="onshore")
ax.plot(monthly.index, monthly["wind_offshore_actual_mw"], label="offshore")
ax.set_title("Monthly mean onshore vs offshore actual generation")
ax.set_ylabel("MW")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIG / "05_onshore_vs_offshore.png", dpi=110)
plt.close(fig)

valid = df.dropna(subset=["wind_onshore_actual_mw", "wind_offshore_actual_mw"])
total = valid["wind_onshore_actual_mw"] + valid["wind_offshore_actual_mw"]
print("onshore share of total: %.1f%%" % (100 * valid["wind_onshore_actual_mw"].sum() / total.sum()))
print("offshore share of total: %.1f%%" % (100 * valid["wind_offshore_actual_mw"].sum() / total.sum()))

onshore share of total: 80.6%
offshore share of total: 19.4%


## 4. Actual wind vs official TSO forecast

In [7]:
sub = df.dropna(subset=["wind_total_actual_mw", "wind_total_forecast_mw"]).copy()
sub["err"] = sub["wind_total_actual_mw"] - sub["wind_total_forecast_mw"]

mae = sub["err"].abs().mean()
rmse = np.sqrt((sub["err"] ** 2).mean())
bias = sub["err"].mean()
nmae = mae / sub["wind_total_actual_mw"].mean()
print(f"n = {len(sub):,}")
print(f"MAE  = {mae:,.1f} MW")
print(f"RMSE = {rmse:,.1f} MW")
print(f"bias = {bias:,.1f} MW")
print(f"Mean-normalized MAE = {nmae:.3f}")

sample = sub.sample(min(20000, len(sub)), random_state=0)
lims = [0, float(max(sub["wind_total_actual_mw"].max(), sub["wind_total_forecast_mw"].max())) * 1.02]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(sample["wind_total_forecast_mw"], sample["wind_total_actual_mw"], s=4, alpha=0.25)
axes[0].plot(lims, lims, "r--", lw=1)
axes[0].set_xlabel("TSO forecast (MW)")
axes[0].set_ylabel("Actual (MW)")
axes[0].set_title("Actual vs TSO forecast")

axes[1].hist(sub["err"], bins=120)
axes[1].axvline(0, color="crimson", lw=1)
axes[1].set_xlabel("error = actual - forecast (MW)")
axes[1].set_ylabel("count")
axes[1].set_title("Forecast error distribution")

fig.tight_layout()
fig.savefig(FIG / "06_actual_vs_forecast.png", dpi=110)
plt.close(fig)

n = 67,339
MAE  = 1,300.9 MW
RMSE = 1,827.0 MW
bias = 97.4 MW
Mean-normalized MAE = 0.088


## 5. Wind generation and electricity price

In [8]:
sub2 = df.dropna(subset=["wind_total_actual_mw", "price_eur_mwh"])
corr = sub2["wind_total_actual_mw"].corr(sub2["price_eur_mwh"])
print("Pearson correlation (actual wind vs day-ahead price): %.3f" % corr)

fig, ax = plt.subplots(figsize=(8, 6))
hb = ax.hexbin(sub2["wind_total_actual_mw"], sub2["price_eur_mwh"], gridsize=55, bins="log", cmap="viridis")
ax.set_xlabel("actual total wind generation (MW)")
ax.set_ylabel("day-ahead price (EUR/MWh)")
ax.set_title("Wind generation vs day-ahead price (log-density)")
fig.colorbar(hb, ax=ax, label="log10(count)")
fig.tight_layout()
fig.savefig(FIG / "07_wind_vs_price.png", dpi=110)
plt.close(fig)

print("\ncorrelation by season:")
for season, g in sub2.groupby(sub2.index.month.map({12:"Winter",1:"Winter",2:"Winter",3:"Spring",4:"Spring",5:"Spring",6:"Summer",7:"Summer",8:"Summer",9:"Autumn",10:"Autumn",11:"Autumn"})):
    print(f"  {season:6s} n={len(g):6,d}  corr={g['wind_total_actual_mw'].corr(g['price_eur_mwh']):.3f}")

Pearson correlation (actual wind vs day-ahead price): -0.260



correlation by season:
  Autumn n=15,430  corr=-0.308
  Spring n=17,664  corr=-0.254
  Summer n=17,663  corr=-0.183
  Winter n=16,585  corr=-0.402


## 6. Missing data and outliers

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df[key_cols].isna().sum().plot.bar(ax=axes[0], color="steelblue")
axes[0].set_title("Missing values per key column")
axes[0].set_ylabel("count")

sns.boxplot(y=df["wind_total_actual_mw"], ax=axes[1], showfliers=False)
axes[1].set_title("Total wind generation distribution (boxplot)")

fig.tight_layout()
fig.savefig(FIG / "08_missing_and_outliers.png", dpi=110)
plt.close(fig)

print(df["wind_total_actual_mw"].describe(percentiles=[.01, .05, .5, .95, .99]).round(1).to_string())

count    67342.0
mean     14853.8
std      10778.1
min         47.0
1%         992.4
5%        2134.6
50%      12113.0
95%      36559.0
99%      43998.5
max      53413.0


## 7. Autocorrelation and seasonality

In [10]:
daily_total = df["wind_total_actual_mw"].resample("D").mean().dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
pd.plotting.autocorrelation_plot(daily_total, ax=axes[0])
axes[0].set_title("Autocorrelation of daily mean wind generation")

lags = list(range(1, 169))
acf = [daily_total.autocorr(lag=l) for l in lags]
axes[1].bar(lags, acf, width=1.0)
axes[1].axhline(0, color="black", lw=0.6)
axes[1].set_xlabel("lag (days)")
axes[1].set_ylabel("autocorrelation")
axes[1].set_title("ACF up to 168 days (1 week)")
fig.tight_layout()
fig.savefig(FIG / "09_autocorrelation.png", dpi=110)
plt.close(fig)

for lag in [1, 2, 3, 7, 14, 30, 90, 365]:
    print(f"ACF lag {lag:3d} days = {daily_total.autocorr(lag=lag):.3f}")

ACF lag   1 days = 0.681
ACF lag   2 days = 0.455
ACF lag   3 days = 0.347
ACF lag   7 days = 0.228
ACF lag  14 days = 0.197
ACF lag  30 days = 0.129
ACF lag  90 days = 0.003
ACF lag 365 days = 0.207


## Summary

- Wind generation is strongly seasonal and autocorrelated; weekly (168 h) lags
  carry substantial information.
- The official TSO forecast is informative but has a measurable error; the error
  distribution is the basis for the Phase 8 benchmark.
- Wind generation is negatively related to day-ahead price (see correlation
  above), but the relationship is non-linear and heteroscedastic.
- Missing values are negligible and documented; negative prices are retained.

See the printed metrics in each section for the exact numbers.

## Modelling availability note (day-ahead, 18:00 Europe/Berlin)

At the 18:00 Europe/Berlin decision time on day D-1, the following are
**genuinely available** for delivery day D:

- Calendar features (day of week, month, hour, holiday flags).
- The **official TSO day-ahead wind forecast** for D (published day-ahead).
- The **day-ahead price** and **load forecast** for D.
- Weather forecast data from a model run initialised before the cutoff.
- **Past actual wind generation** strictly before the decision time (used by the
  decision-time lag/rolling features).

The following are **not** available at decision time and must not be used as
features: contemporaneous actual wind generation (the target), future actuals,
and the realised day-ahead price. The processed dataset's `decision_time_utc`
column encodes the cutoff, and the lag features are masked against it.